THEME: This project automatically collects the latest news from trusted RSS sources like BBC, CNN, and TechCrunch, converts them into numerical embeddings, and stores them in a vector database. Users can search for topics, get personalized article recommendations, and generate AI-based summaries using a local language model—no API key required.

# Install

gradio → Creates an interactive web-based user interface for the news recommender.

transformers → Provides pre-trained language models for text summarization and processing.

sentencepiece → Helps tokenize text efficiently for transformer-based models.

sentence-transformers → Generates embeddings for news articles to enable similarity search.

chromadb → Stores and retrieves article embeddings using a vector database.

feedparser → Reads and extracts news data from RSS feeds of various websites.

torch → Powers deep learning models used for embeddings and text generation.

In [ ]:

!pip install -q gradio transformers sentencepiece sentence-transformers chromadb feedparser torch

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.0 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.6/21.6 MB 44.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 81.5/81.5 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 103.3/103.3 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.4/17.4 MB 53.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.5/72.5 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 132.4/132.4 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.4/66.4 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

Imports Python libraries needed for time tracking, logging, threading, and handling news data.

In [ ]:
import time
import threading
import uuid
import logging
from datetime import datetime
from typing import List, Dict

import feedparser
import pandas as pd
import numpy as np
import gradio as gr

# embeddings & LLM
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

# Vector DB
import chromadb
from chromadb.config import Settings

# news sources

These are the some popular new channels like BBC, CNN, Nytimes, Techchrunch

In [ ]:

RSS_FEEDS = [
    "http://feeds.bbci.co.uk/news/rss.xml",
    "http://rss.cnn.com/rss/edition.rss",
    "https://feeds.npr.org/1001/rss.xml",
    "https://rss.nytimes.com/services/xml/rss/nyt/HomePage.xml",
    "https://techcrunch.com/feed/",
]


    # add more feeds you like

AUTO_FETCH_INTERVAL_SECONDS = 60 * 3  # fetch every 3 minutes (change as desired)
CHROMA_PERSIST_DIR = "./news_chroma_store"  # set to a path to persist, e.g. "./chroma_db"

EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
LLM_MODEL_NAME = "google/flan-t5-base"  # used for summaries / re-rank / explanation

TOP_K_DEFAULT = 5

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("real_time_news_recommender")

These URLs are official news websites. Based on these sources, we will fetch and display the news. You can also add more sites if you want.


NOTE

 When you ask something related to the content available on those news sources, the system will process your request; otherwise, it won’t execute.

# Configuration & Model Loading

Loads SentenceTransformer to create embeddings for articles and Flan-T5 to summarize news.

Initializes ChromaDB to store articles and their vector representations persistently.

In [ ]:
logger.info("Loading embedding model...")
embedder = SentenceTransformer(EMBEDDING_MODEL_NAME)

logger.info("Loading LLM (Flan-T5)...")
tokenizer = AutoTokenizer.from_pretrained(LLM_MODEL_NAME)
llm = AutoModelForSeq2SeqLM.from_pretrained(LLM_MODEL_NAME, device_map="cpu", torch_dtype=None)

logger.info("Initializing ChromaDB client...")
chroma_client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)

COLLECTION_NAME = "news_articles"
if COLLECTION_NAME in [c.name for c in chroma_client.list_collections()]:
    collection = chroma_client.get_or_create_collection("news_articles")
else:
    collection = chroma_client.create_collection(name=COLLECTION_NAME, metadata={"created_at": str(datetime.utcnow())})

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/990M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

/tmp/ipython-input-3181908494.py:15: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  collection = chroma_client.create_collection(name=COLLECTION_NAME, metadata={"created_at": str(datetime.utcnow())})


Utilities: fetch, clean, embed, insert

Step 1: Fetch news from RSS websites

Step 2: Remove duplicates

Step 3: Convert news into embeddings

Step 4: Save all news inside

Step 5: This runs automatically

In [ ]:
def fetch_feed_entries(feeds: List[str]) -> List[Dict]:
    """Fetch latest entries from RSS feeds and return as list of dicts."""
    articles = []
    for feed_url in feeds:
        try:
            parsed = feedparser.parse(feed_url)
            for entry in parsed.entries:
                # create standardized record
                title = entry.get("title", "")
                summary = entry.get("summary", "") or entry.get("description", "")
                link = entry.get("link", "")
                published = entry.get("published", entry.get("updated", ""))
                uid = entry.get("id") or link or str(uuid.uuid4())
                # inside summary sometimes html — keep raw for now or strip if needed
                articles.append({
                    "id": uid,
                    "title": title,
                    "summary": summary,
                    "link": link,
                    "published": published,
                    "source": feed_url
                })
        except Exception as e:
            logger.warning(f"Failed to fetch {feed_url}: {e}")
    return articles


def dedupe_new_articles(articles: List[Dict]) -> List[Dict]:
    """Remove articles already present in Chroma by checking IDs in metadata (simple approach)."""
    if not articles:
        return []
    existing_ids = set(collection.get()['ids']) if collection.count() > 0 else set()
    new = [a for a in articles if a["id"] not in existing_ids]
    return new


def embed_texts(texts: List[str]) -> np.ndarray:
    """Get embeddings for a list of texts (returns numpy array)."""
    if not texts:
        return np.array([])
    emb = embedder.encode(texts, convert_to_numpy=True, show_progress_bar=False)
    return emb


def insert_articles_into_chroma(articles: List[Dict]):
    """Insert a batch of articles into Chroma with embeddings and metadata."""
    if not articles:
        return 0
    texts_for_embedding = [f"{a['title']}\n\n{a['summary']}" for a in articles]
    embeddings = embed_texts(texts_for_embedding).tolist()
    ids = [a["id"] for a in articles]
    metadatas = [
        {
            "title": a["title"],
            "summary": a["summary"],
            "link": a["link"],
            "published": a["published"],
            "source": a["source"]
        } for a in articles
    ]
    # insert into chroma
    collection.add(ids=ids, embeddings=embeddings, metadatas=metadatas, documents=texts_for_embedding)
    logger.info(f"Inserted {len(articles)} new articles into Chroma.")
    if CHROMA_PERSIST_DIR:
        chroma_client.persist()
    return len(articles)


# Recommender and LLM helper

search_recommend() → Finds and returns top news articles that closely match the user’s query.

llm_summarize() → Generates a short, clear summary of chosen news articles using an AI model.

In [ ]:
def search_recommend(query: str, top_k: int = TOP_K_DEFAULT):
    """Return top-k similar articles (embedding similarity) and optional LLM summary."""
    if not query:
        return pd.DataFrame([{"error": "Please enter a query or select a category."}])

    # create query embedding
    query_emb = embed_texts([query])[0].tolist()

    # search chroma
    try:
        results = collection.query(query_embeddings=[query_emb], n_results=top_k, include=['metadatas', 'distances', 'documents'])
    except Exception as e:
        logger.error(f"Chroma query failed: {e}")
        return pd.DataFrame([{"error": "Search failed (no articles yet or DB error)."}])

    # results structure: list per query
    docs = []
    metadatas = results.get("metadatas", [[]])[0]
    distances = results.get("distances", [[]])[0]
    documents = results.get("documents", [[]])[0]

    for i, meta in enumerate(metadatas):
        docs.append({
            "rank": i + 1,
            "title": meta.get("title"),
            "summary": meta.get("summary"),
            "link": meta.get("link"),
            "published": meta.get("published"),
            "source": meta.get("source"),
            "distance": float(distances[i]) if i < len(distances) else None
        })

    if not docs:
        return pd.DataFrame([{"Result": "No matches found. Try changing the query or refresh the news."}])

    df = pd.DataFrame(docs)
    return df


def llm_summarize(texts: List[str], max_new_tokens: int = 120):
    """Use local Flan-T5 to generate a short summary or re-rank explanation for a list of texts."""
    # Compose prompt
    joined = "\n\n---\n\n".join(texts)
    prompt = f"Summarize the following news articles in 3 bullet points highlighting the most important facts and why they matter:\n\n{joined}\n\nSummary:"
    inputs = tokenizer(prompt, return_tensors="pt", truncation=True, max_length=1024)
    outputs = llm.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    reply = tokenizer.decode(outputs[0], skip_special_tokens=True)
    return reply

# Real-time fetch loop



auto_fetch_loop() → Continuously runs the fetch process at fixed intervals without user intervention.

start_auto_fetcher() → Launches a background thread that begins the automated news-fetching cycle.

In [ ]:
_fetch_lock = threading.Lock()
_last_fetch_time = None
_last_fetch_count = 0



def fetch_and_update_once():
    global _last_fetch_time, _last_fetch_count
    with _fetch_lock:
        logger.info("Fetching RSS feeds...")
        all_articles = fetch_feed_entries(RSS_FEEDS)
        new = dedupe_new_articles(all_articles)
        inserted = insert_articles_into_chroma(new)
        _last_fetch_time = datetime.utcnow()
        _last_fetch_count = inserted
        logger.info(f"Fetch complete. Inserted {inserted} articles.")


def auto_fetch_loop(interval_seconds: int):
    logger.info("Starting auto-fetch loop (daemon thread).")
    while True:
        try:
            fetch_and_update_once()
        except Exception as e:
            logger.error(f"Auto-fetch error: {e}")
        time.sleep(interval_seconds)


# Start daemon thread for auto fetching (only once)
_auto_thread = None
def start_auto_fetcher(interval_seconds: int = AUTO_FETCH_INTERVAL_SECONDS):
    global _auto_thread
    if _auto_thread is None:
        t = threading.Thread(target=auto_fetch_loop, args=(interval_seconds,), daemon=True, name="news-auto-fetcher")
        t.start()
        _auto_thread = t
        logger.info("Auto fetcher started.")

# Gradio UI
Builds an interactive interface for searching, refreshing, and summarizing news.

Allows users to ask questions and receive answers powered by the stored articles.

In [ ]:
def manual_refresh():
    """Manual refresh button callback."""
    try:
        fetch_and_update_once()
        status = f"Manual refresh done at {_last_fetch_time} (inserted {_last_fetch_count} new articles)."
    except Exception as e:
        status = f"Manual refresh failed: {e}"
    return status


def recommend_callback(user_query, top_k):
    df = search_recommend(user_query, top_k)
    return df


def summarize_selected_article(article_link_or_index):
    """
    Accepts either a link or an integer index for demonstration.
    We'll find the matching doc in the collection metadata and summarize its content.
    """
    # find by id/link
    all_meta = collection.get(include=["metadatas", "documents"])
    metadatas = all_meta.get("metadatas", [])
    documents = all_meta.get("documents", [])
    # simple lookup by link
    for i, meta in enumerate(metadatas):
        if meta.get("link") == article_link_or_index or meta.get("title") == article_link_or_index:
            doc_text = documents[i]
            summary = llm_summarize([doc_text])
            return summary
    return "Article not found in DB. Try selecting an article from recommendations."


with gr.Blocks(theme=gr.themes.Soft()) as demo:
    gr.Markdown("## 📰 Real-Time News Recommender — (Free models, no API keys)")

    # top-row controls
    with gr.Row():
        refresh_btn = gr.Button("Manual Refresh Now 🔁")
        status_box = gr.Textbox(label="Last Refresh Status", value="Not fetched yet", interactive=False)
        start_auto_btn = gr.Button("Start Auto-Fetch (daemon)")
        interval_box = gr.Number(label="Auto Fetch Interval (seconds)", value=AUTO_FETCH_INTERVAL_SECONDS)

    with gr.Tabs():

        with gr.Tab("News Recommender"):
            gr.Markdown("### Search or type a topic (e.g., 'AI', 'India politics', 'cricket', 'bitcoin')")

            query = gr.Textbox(label="Query / Topic")
            top_k = gr.Slider(minimum=1, maximum=20, value=5, step=1, label="Top K results")
            search_btn = gr.Button("Search 🔎")
            results_df = gr.Dataframe(interactive=False, label="Recommendations (title, summary, link, source)")

            search_btn.click(fn=recommend_callback, inputs=[query, top_k], outputs=[results_df])

            gr.Markdown("Select an article title (or paste link) then click summarize:")
            article_input = gr.Textbox(label="Article Title or Link")
            summarize_btn = gr.Button("Summarize Selected Article (LLM)")
            summary_output = gr.Textbox(label="Summary / Explanation")
            summarize_btn.click(fn=summarize_selected_article, inputs=article_input, outputs=summary_output)


        with gr.Tab("AI Assistant"):
            gr.Markdown("### Ask the assistant to summarize recent trending news or explain why something matters.")
            ask_box = gr.Textbox(label="Prompt (e.g., 'Summarize latest AI news')", lines=3)
            ask_btn = gr.Button("Ask AI")
            ai_answer = gr.Textbox(label="AI Answer (LLM)")

            def ai_assist_fn(prompt_text):
                # We'll gather top 5 latest docs and give LLM context + prompt
                all_docs = collection.get(include=["metadatas", "documents"])
                docs = all_docs.get("documents", [])[:5]
                if not docs:
                    return "No articles available yet. Please refresh the news."
                summary = llm_summarize(docs + [f"User question: {prompt_text}"], max_new_tokens=160)
                return summary

            ask_btn.click(fn=ai_assist_fn, inputs=ask_box, outputs=ai_answer)

    # callbacks for top-row controls
    def on_manual_refresh_click():
        return manual_refresh()

    def on_start_auto_fetch_click(interval_seconds):
        try:
            start_auto_fetcher(int(interval_seconds))
            return f"Auto fetch started with interval {int(interval_seconds)} seconds. Running in background."
        except Exception as e:
            return f"Failed to start auto-fetch: {e}"

    refresh_btn.click(fn=on_manual_refresh_click, inputs=None, outputs=status_box)
    start_auto_btn.click(fn=on_start_auto_fetch_click, inputs=interval_box, outputs=status_box)

# Launch (share True for external access)
if __name__ == "__main__":
    # Optionally run one immediate fetch so UI has content
    try:
        fetch_and_update_once()
    except Exception as e:
        logger.warning(f"Initial fetch failed: {e}")

    demo.launch(share=True)

/tmp/ipython-input-317042305.py:34: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft()) as demo:


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://e3b1772a88c68c2aba.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
